## Classification for explosive-media-footage notebook-1 

#### All telegram messages were scraped from Jan 01, 2026 till April 09, 2026. the Iran-US war. The first attack took place on Feb, 28, 2026. The purpose of this exercise is to see how the messaging on this Telegram channel changed over time, comparing the before with after the war. And visualizing this comparison in a narrative scrollytelling format. 

#### Telegram messages were scraped using TelegramAPI. The scraper ran twice, first on April 02, 2026 and then on April 11, 2026. 

#### Once the scraping was done I had the following columns:

 - message_text_persian
 - message_text_english
 - time_est
 - date
 - has_media
 - media_filename
 - audio_transcriptions_persian
 - audio_transcriptions_english
 - ocr_text_persian
 - ocr_text_english
 - keywords
 - screenshots
 - theme
 - include_person
 - AI_generated

#### From the scraper I got input for columns 1,3,4,5,6. 

#### Follow notebooks 2, and 3 for the entire process: transcription, translation, and classification.

In [17]:
#importing what i need to make this system work 

import pandas as pd
import csv, requests, os
import numpy as np
import sys 

In [7]:
def make_regular_gsheet_url(doc_id, sheet_id):
    return f"https://docs.google.com/spreadsheets/d/{doc_id}/edit#gid={sheet_id}"

def make_csv_gsheet_url(doc_id, sheet_id):
    return f"https://docs.google.com/spreadsheets/d/{doc_id}/export?format=csv&id={doc_id}&gid={sheet_id}"


In [ ]:
#connecting my google sheet to this notebook 

GOOGLE_SHEET_ID = '1Gz7KgWNzbAcUG5R8LmGZ7lRn41orQKnbEzTc-mt6Zb0'  
SHEET_GID = '1212652360'                       


google_sheet_url = make_regular_gsheet_url(GOOGLE_SHEET_ID, SHEET_GID)
print("Querying Doc:", google_sheet_url)

google_sheet_csv_url = make_csv_gsheet_url(GOOGLE_SHEET_ID, SHEET_GID)
response = requests.get(google_sheet_csv_url)
response.raise_for_status()

reader = csv.reader(response.text.splitlines())
header = next(reader)
df = pd.DataFrame(list(reader), columns=header)

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(df.columns.tolist())
df.head()

Querying Doc: https://docs.google.com/spreadsheets/d/1Gz7KgWNzbAcUG5R8LmGZ7lRn41orQKnbEzTc-mt6Zb0/edit#gid=1212652360
Loaded 2840 rows, 15 columns
['message_text_persian', 'message_text_english', 'time_est', 'date', 'has_media', 'media_filename', 'audio_transcription_persian', 'audio_transcription_english', 'ocr_text_persian', 'ocr_text_english', 'keywords', 'screenshots', 'theme', 'include_person', 'AI_generated']


,message_text_persian,message_text_english,time_est,date,has_media,media_filename,audio_transcription_persian,audio_transcription_english,ocr_text_persian,ocr_text_english,keywords,screenshots,theme,include_person,AI_generated
0,ØµØ¨Ø­ØªÙÙ Ø¨Ø®ÛØ± ðð ð @akhbarenf...,,2025-12-31 22:58:12 EST,2025-12-31,Y,2025-12-31_001.mp4,,,,,,,,,
1,ÚÛ ÙÛØ®ÙØ§Ø³ØªÛÙØ ÚÛ Ø´Ø¯!!!ð£ðºï¸...,,2025-12-31 23:46:46 EST,2025-12-31,Y,2025-12-31_002.jpg,,,Ø§ÙÛØ±Ø­Ø³Ø§Ù Ø®Ø¯Ø§ÛØ§Ø±ÛÙØ±Ø¯ÙØªÙÙØ¯ Û...,Amir Hossam Khodayari FardBorn 1382 - Martyred...,"Protest, Martyrdom, Unrest, Iran, Public Order",,,,
2,ð Ø³Ø±ÙØ§ Ø§ÙØ±ÙØ² ÙÛØ±ÙØ ÙØ±Ø¯Ø§ Ø¨Ø±...,,2026-01-01 00:00:20 EST,2026-01-01,Y,2026-01-01_001.jpg,,,Ø¯Ø±Ø§Û Ø®Ø²Ø±Ø¯Ø±ÛØ§Û Ø¹ÙØ§ÙØ®ÙÛØ¬ ÙØ§...,Caspian SeaSea of OmanPersian Gulf,,,,,
3,ð ÙØ·Ø§ÙØ¨Ù Ø¢Ø±ÙØ Ø¨Ø§Ø²Û ØªÙ Ø²ÙÛÙ...,,2026-01-01 00:13:22 EST,2026-01-01,Y,2026-01-01_002.mp4,,,,,,,,,
4,ð Ø­ÙÙÙ Ø¢ÙØ±ÛÚ©Ø§ Ø¨Ù Û³ ÙØ§ÛÙ Ø¯Û...,,2026-01-01 00:20:49 EST,2026-01-01,Y,2026-01-01_003.mp4,,,,,,,,,


In [10]:
#translating the captions using Haiku 4.5 in the local CSV and then reimporting it to the Google Sheet
#honestly im doing this bec i don't have excel, and i want to see what my data is looking like after each step and do that not in the ew csv format
#so if you're someone who is trying to replicate this entire process, you can just skip the connecting-to-google-sheet step and directly read your local csv

import anthropic, os, time, json, pandas as pd
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
CSV_FILE = "explosive_media_messages.csv"
BATCH_SIZE = 10
SAVE_EVERY = 50

df = pd.read_csv(CSV_FILE)

# find rows that need translation
needs_translation = []
for idx in df.index:
    persian = str(df.at[idx, "message_text_persian"]).strip()
    english = str(df.at[idx, "message_text_english"]).strip()
    if persian and persian != "nan" and english in ("", "nan", "[translation failed]"):
        needs_translation.append(idx)

print(f"{len(needs_translation)} rows need translation")

translated_count = 0

for batch_start in range(0, len(needs_translation), BATCH_SIZE):
    batch_indices = needs_translation[batch_start:batch_start + BATCH_SIZE]
    
    # build numbered list of texts
    numbered_texts = []
    for i, idx in enumerate(batch_indices, 1):
        text = str(df.at[idx, "message_text_persian"]).strip()
        numbered_texts.append(f"[{i}] {text}")
    
    prompt = "Translate each numbered Persian/Farsi text below to English. Return ONLY a JSON array of strings, where each string is the English translation in order. No extra text.\n\n" + "\n\n".join(numbered_texts)
    
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=4096,
            messages=[{"role": "user", "content": prompt}]
        )
        
        raw = response.content[0].text.strip()
        # parse JSON array from response
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        translations = json.loads(raw)
        
        for i, idx in enumerate(batch_indices):
            if i < len(translations):
                df.at[idx, "message_text_english"] = translations[i]
                translated_count += 1
            else:
                df.at[idx, "message_text_english"] = "[translation failed]"
    except Exception as e:
        print(f"  [batch error at row {batch_indices[0]}]: {e}")
        for idx in batch_indices:
            df.at[idx, "message_text_english"] = "[translation failed]"
    
    # progress + periodic save
    if translated_count % SAVE_EVERY < BATCH_SIZE:
        print(f"  [{translated_count}/{len(needs_translation)}] translated")
        df.to_csv(CSV_FILE, index=False)
    
    time.sleep(0.2)  # rate limit buffer

# final save
df.to_csv(CSV_FILE, index=False)
print(f"Done. Translated {translated_count} rows.")


2790 rows need translation
  [50/2790] translated
  [100/2790] translated
  [150/2790] translated
  [200/2790] translated
  [250/2790] translated
  [300/2790] translated
  [350/2790] translated
  [400/2790] translated
  [450/2790] translated
  [500/2790] translated
  [550/2790] translated
  [600/2790] translated
  [650/2790] translated
  [700/2790] translated
  [750/2790] translated
  [800/2790] translated
  [850/2790] translated
  [900/2790] translated
  [950/2790] translated
  [1000/2790] translated
  [1050/2790] translated
  [1100/2790] translated
  [1150/2790] translated
  [1200/2790] translated
  [1250/2790] translated
  [1300/2790] translated
  [1350/2790] translated
  [1400/2790] translated
  [1450/2790] translated
  [1500/2790] translated
  [1550/2790] translated
  [1600/2790] translated
  [1650/2790] translated
  [1700/2790] translated
  [1750/2790] translated
  [1800/2790] translated
  [1850/2790] translated
  [1900/2790] translated
  [1950/2790] translated
  [2000/2790] tran

In [11]:
df.head(5)

,message_text_persian,message_text_english,time_est,date,has_media,media_filename,audio_transcription_persian,audio_transcription_english,ocr_text_persian,ocr_text_english,keywords,screenshots,theme,include_person,AI_generated
0,صبحتون بخیر 👋😁 \n\n🆔 @akhbarenfejari,Good morning 👋😁,2025-12-31 22:58:12 EST,2025-12-31,Y,2025-12-31_001.mp4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,چی میخواستیم، چی شد!!!😣\n\n🔺️معاون سیاسی استان...,"What we wanted, what happened!!! 😣 The politic...",2025-12-31 23:46:46 EST,2025-12-31,Y,2025-12-31_002.jpg,NaN,NaN,امیرحسام خدایاریفرد\nمتولد ۱۳۸۲ - شهادت در کوه...,Amir Hossam Khodayari Fard\nBorn 1382 - Martyr...,"Protest, Martyrdom, Unrest, Iran, Public Order",NaN,NaN,NaN,NaN
2,🛑 سرما امروز میره، فردا برمی‌گرده\n\n🌨امروز تو...,"🛑 Cold is leaving today, coming back tomorrow....",2026-01-01 00:00:20 EST,2026-01-01,Y,2026-01-01_001.jpg,NaN,NaN,درای خزر\nدریای عمان\nخلیج فارس,Caspian Sea\nSea of Oman\nPersian Gulf,NaN,NaN,NaN,NaN,NaN
3,🛑 مطالبه آره، بازی تو زمین دشمن نه\n\n🔺️دکتر ر...,"🛑 Demands yes, playing on enemy territory no. ...",2026-01-01 00:13:22 EST,2026-01-01,Y,2026-01-01_002.mp4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,🛑 حمله آمریکا به ۳ قایق دیگه\n\n💣 آمریکا دوبار...,🛑 America attacks 3 more boats. America again ...,2026-01-01 00:20:49 EST,2026-01-01,Y,2026-01-01_003.mp4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### ran the take_screenshots.py script in a new terminal to create a folder full of screenshots of videos 
#### putting them in the screenshots folder, and populating them on the CSV in the relevant column

! .venv/bin/python take_screenshots.py

#### the script returned this result: Done. Processed: 981, Skipped: 0, Errors: 0 --> the dataset includes 981 videos among 2,841 Telegram messages

##### each .mp4 file in the scraped-media folder contains an audio and a video. i used ffmpeg to strip out the audio as a .wav file at 16kHz mono (the format whisper expects) to transcribe the persian. each audio file was broken into 28-second chunks and then joined together to fill the cells in the audio_transcription_persian column. used haiku 4.5 to translate the persian transcriptions to english.

#### refer to notebook 2 for the whisper-persian workflow